# EDA and Preprocessing

## 1. Nạp dữ liệu

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/VietJobs_IT.csv")

## 2. Khám phá dữ liệu (EDA)

### 2.1 Kích thước tập dữ liệu

In [2]:
print(df.shape)

(1906, 18)


In [3]:
print(df.head(5).to_string())

                                   job_title     location   country                                                                            qualifications                                                                                                                                                technical_skills                                                                                                                                      soft_skills languages_required experience_required             salary   contract_type                                           working_hours                                                                                                                                                                                                                                                                                                                       benefits                                                                                         

### 2.2 Kiểu dữ liệu của các cột

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1906 entries, 0 to 1905
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   job_title            1906 non-null   str  
 1   location             1906 non-null   str  
 2   country              1906 non-null   str  
 3   qualifications       1901 non-null   str  
 4   technical_skills     1804 non-null   str  
 5   soft_skills          1799 non-null   str  
 6   languages_required   634 non-null    str  
 7   experience_required  1906 non-null   str  
 8   salary               1906 non-null   str  
 9   contract_type        1906 non-null   str  
 10  working_hours        1783 non-null   str  
 11  benefits             1905 non-null   str  
 12  description          1906 non-null   str  
 13  requirements_text    1906 non-null   str  
 14  category             1906 non-null   str  
 15  salary_min           1906 non-null   int64
 16  salary_max           1906 non-null 

### 2.3 Tỷ lệ giá trị thiếu theo cột

In [5]:
missing_stats = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": round(df.isna().mean()*100,2)
})
missing_stats = missing_stats.sort_values(
    by="missing_pct",
    ascending= False
)
missing_stats

,missing_count,missing_pct
languages_required,1272,66.74
working_hours,123,6.45
soft_skills,107,5.61
technical_skills,102,5.35
qualifications,5,0.26
benefits,1,0.05
location,0,0.00
job_title,0,0.00
country,0,0.00
experience_required,0,0.00


### 2.4 Phân bố chức danh công việc

In [6]:
df["job_title"].value_counts()


job_title
Nhân Viên Thiết Kế                          43
Thiết kế đồ họa                             30
Nhân Viên Thiết Kế Đồ Họa                   27
Nhân Viên Kinh Doanh                        25
Nhân Viên Kinh Doanh Phần Mềm               25
                                            ..
2D Background Artist                         1
Delivery Manager (PM Kiêm Presale)           1
R&D AI Analyst – Ứng Dụng AI                 1
Trưởng nhóm (.NET)                           1
Nhân Viên Kinh Doanh - Lĩnh Vực Phần Mềm     1
Name: count, Length: 1328, dtype: int64

In [7]:
df["job_title"].nunique()

1328

In [8]:
df["category"].value_counts()

category
công_nghệ_thông_tin_kỹ_thuật_số    1906
Name: count, dtype: int64

**Nhận xét** </br>
Bộ dữ liệu thuộc nhóm Công nghệ Thông tin và Kỹ thuật số. Mặc dù xuất hiện một số vị trí thuộc lĩnh vực thiết kế hoặc nghiệp vụ, phần lớn các bản ghi vẫn liên quan đến CNTT, phát triển phần mềm, hạ tầng hệ thống và dữ liệu. </br>
Số lượng job title rất lớn (1328 giá trị khác nhau trên 1906 bản ghi), cho thấy dữ liệu có mức độ đa dạng cao và tồn tại nhiều biến thể tên gọi cho cùng một nhóm nghề nghiệp. Do đó ở giai đoạn xử lý cần gom nhóm về các role chính

### 2.5 Xem nhanh các kỹ năng kỹ thuật

In [9]:
df["technical_skills"].value_counts()

technical_skills
['Photoshop', 'Illustrator']                                                                                                                                22
['Adobe Photoshop', 'Illustrator', 'InDesign']                                                                                                               9
['Photoshop', 'Illustrator', 'Indesign']                                                                                                                     6
['Adobe Illustrator', 'Adobe Photoshop']                                                                                                                     5
['Adobe Photoshop', 'Adobe Illustrator', 'Adobe InDesign']                                                                                                   5
                                                                                                                                                            ..
['HTML', 'CSS', 'JavaScript (

**Nhận xét**
Tương tự job title, technical skills cũng có rất nhiều các skill giống nhau nhưng khác tên, cần chuẩn hóa và gom nhóm lại

### 2.6 Khai phá danh sách kỹ năng

In [10]:
import ast
#đưa chuỗi về list
def parse_list(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float)) or str(x).strip() in ("", "nan"):
        return []
    try:
        v = ast.literal_eval(x)
        return v if isinstance(v, list) else []
    except Exception:
        return []

In [11]:
df["skills_list"] = df["technical_skills"].apply(parse_list)
type(df["skills_list"].iloc[0])

list

In [12]:
df["skills_list"].apply(len).describe() #xem trung bình 1 job có bao nhiêu skill

count    1906.000000
mean        5.083945
std         4.148210
min         0.000000
25%         3.000000
50%         4.000000
75%         6.000000
max        39.000000
Name: skills_list, dtype: float64

In [13]:
from collections import Counter

all_skills = [] # lưu tất cả các skill có trong technical_skills

for skills in df["skills_list"]:
    all_skills.extend(skills)

skill_counts = Counter(all_skills)

In [14]:
skill_counts

Counter({'Photoshop': 221,
         'Illustrator': 167,
         'Python': 161,
         'Java': 141,
         'SQL': 117,
         'Docker': 105,
         'Git': 103,
         'Figma': 99,
         'Adobe Photoshop': 96,
         'MySQL': 94,
         'JavaScript': 87,
         'Adobe Illustrator': 87,
         'CSS': 82,
         'HTML': 81,
         'PostgreSQL': 78,
         'Kubernetes': 66,
         'C#': 64,
         'Oracle': 63,
         'MongoDB': 61,
         'InDesign': 61,
         'AWS': 59,
         'PHP': 52,
         'AI': 49,
         'ReactJS': 46,
         'Jira': 46,
         'After Effects': 43,
         'SQL Server': 41,
         'Linux': 41,
         'Angular': 40,
         'CI/CD': 38,
         'Premiere': 38,
         'Redis': 35,
         'Javascript': 35,
         'Kafka': 34,
         'Spring Boot': 34,
         'Azure': 32,
         'Excel': 32,
         'Node.js': 30,
         'RESTful API': 28,
         'After Effect': 28,
         'Jenkins': 28,
       

In [15]:
skill_counts.most_common(20)

[('Photoshop', 221),
 ('Illustrator', 167),
 ('Python', 161),
 ('Java', 141),
 ('SQL', 117),
 ('Docker', 105),
 ('Git', 103),
 ('Figma', 99),
 ('Adobe Photoshop', 96),
 ('MySQL', 94),
 ('JavaScript', 87),
 ('Adobe Illustrator', 87),
 ('CSS', 82),
 ('HTML', 81),
 ('PostgreSQL', 78),
 ('Kubernetes', 66),
 ('C#', 64),
 ('Oracle', 63),
 ('MongoDB', 61),
 ('InDesign', 61)]

Kiểm tra các kỹ năng khác nhau

In [16]:
len(all_skills) #trùng

9690

In [17]:
print("Các skill được nêu trong dataset: ",len(set(all_skills))) # không trùng

Các skill được nêu trong dataset:  3771


Kiểm tra các kỹ năng trùng nghĩa nhưng khác cách viết

In [18]:
unique_skills = sorted(set(all_skills))

print(unique_skills[:200])

['.NET', '.NET (C#, ASP.NET, .NET Core)', '.NET C#', '.NET Core', '.NET Core MVC', '.NET Framework', '.NET MVC', '.NET/C#', '.NETCore', '.Net', '.Net (C#, VB.NET)', '.Net Core', '.Net core', '.NetCore', '2+ years of experience as a DevOps Engineer or similar role', '2D Animation', '2D Animation (traditional & Digital)', '2D Art', '3D', '3D Animation', '3D Artist', '3D Coat', '3D Max', '3D Modeling', '3D libraries', '3D max', '3D modeling', '3DMAX', '3DMax', '3DSMAX', '3DSmax', '3Dmax', '3Ds Max', '3DsMax', '3d Modeling', '3ds Max', '3dsMax', '3dsmax', '7 QC tools trong SPC', 'A/B testing', 'ABAP', 'ABAP Restful web services', 'ACL', 'AD', 'AD/Email', 'ADF', 'AE', 'AI', 'AI (Azure OpenAI and GenAI)', 'AI (ChatGPT, Gemini)', 'AI (MJ, Kling...)', 'AI (Midjourney, Stable Diffusion, Kling...)', 'AI / Machine Learning', 'AI Thiết kế', 'AI application in work', 'AI caption/image tool', 'AI hỗ trợ sáng tạo (ChatGPT, DALL·E)', 'AI photo (Leonardo, RunWay, Midjourney)', 'AI practices: Github Cop

Ghi ra file tên các kỹ năng khác nhau trong dataset từ đó gom nhóm các kỹ năng cần chuẩn hóa

In [19]:
unique_skills = sorted(set(all_skills))

with open("../data/processed/unique_skills.txt", "w", encoding="utf-8") as f: #ghi ra file
    for skill in unique_skills:
        f.write(skill + "\n")

## 3. Tiền xử lý dữ liệu

### 3.1 Xây dựng tập chứa từ khóa các kỹ năng mềm và các kỹ năng chung chung
Tạo tập soft_prefix và genaric dựa vào việc dùng phần mềm scan file unique_skils.txt, vẫn còn sót nhiều và cần canh chỉnh nhiều lần

In [20]:
import re
SOFT_PREFIX = re.compile(
    r"^(kỹ năng|khả năng|có khả năng|kinh nghiệm|hiểu biết|kiến thức|tư duy|năng lực|thành thạo|"
    r"communication|teamwork|good |strong |ability|experience|understanding|knowledge of|proficien)"
    , re.I
)

In [21]:
#Hàm kiểm tra 1 kỹ năng có phải kĩ năng mềm không
def is_soft_skill(skill_name):
    if SOFT_PREFIX.match(skill_name.strip()):
        return True
    else: return False

#test
print(is_soft_skill("Kỹ năng giao tiếp"))
print(is_soft_skill("Java"))

True
False


In [22]:
GENERIC = {
    "phần mềm", "hệ điều hành", "hệ điều hành windows", "windows", "lập trình", "system", "internet",
    "công nghệ thông tin", "phát triển phần mềm", "phần cứng", "web",
    "tiếng nhật", "tiếng anh", "giải pháp cntt",
    "kiến trúc phần mềm", "thiết kế hệ thống", "xml", "json", "http", "ai tools",
}

In [23]:
# ngoài ra phần chữ ngắn dễ nhầm khi dò substring -> chỉ khớp nguyên chuỗi, nên cần safe_short để tăng độ chính xác
SAFE_SHORT = {"sql", "css", "aws", "git", "php", "gcp", "xml", "jwt", "oop", "etl", "erp", "crm", "uml",
              "npm", "vpn", "dns", "nlp", "c#", "c++", "ci/cd", "tcp/ip", ".net", "html", "kvm", "nas"}

**Lưu ý**
Trước khi tiến hành xóa cần đưa dữ liệu từ dạng string về list, để các hàm trong Python có thể hoạt động 1 cách chính xác và nhanh chóng nhất vì dễ dàng nhận biết được các skill là các phần tử độc lập thay vì phải đọc nguyên đoạn chuỗi dài

### 3.2 Chuẩn hóa danh sách technical_skills
Đã có hàm chuyển string thành list ở mục 2.6 nhưng vẫn chưa thống nhất từng list sẽ được định dạng như thế nào nên ở mục này bước đầu tiên sẽ là đưa về list, sau đó chuẩn hóa list


##### Bước 1: Chuẩn hóa từng skill

Bỏ khoảng trắng thừa và không phân biệt chữ hoa, chữ thường khi tra cứu. Tên hiển thị chuẩn vẫn được giữ theo dạng dễ đọc.


In [24]:
#Tạo khóa so khớp nhất quán, không làm thay đổi tên hiển thị.
def key(skill):
    if not isinstance(skill, str):
        return ""
    return re.sub(r"\s+", " ", skill).strip().casefold()


##### Bước 2: Khai báo các tên đồng nghĩa

Mỗi khóa trong `ALIAS_GROUPS` là tên chuẩn; danh sách đi kèm là những cách viết được quy về tên đó. Cần rà soát từ điển theo dữ liệu thực tế và chỉ gộp các tên thực sự cùng nghĩa.


In [25]:
ALIAS_GROUPS = {
    # --- Design / Creative
    "Photoshop": ["photoshop", "adobe photoshop", "ps"],
    "Illustrator": ["illustrator", "adobe illustrator"],
    "InDesign": ["indesign", "adobe indesign"],
    "After Effects": ["after effects", "after effect", "adobe after effects", "adobe after effect", "ae"],
    "Premiere": ["premiere", "adobe premiere", "premiere pro", "premier pro", "adobe premiere pro"],
    "CorelDRAW": ["corel", "coreldraw", "corel draw"],
    "3ds Max": ["3ds max", "3dsmax", "3d max", "3dmax"],
    "AutoCAD": ["autocad", "auto cad"],
    "Canva": ["canva", "canva pro"],
    "Unreal Engine": ["unreal engine", "unreal engine 5"],
    "Adobe XD": ["adobe xd"],
    "UI/UX": ["ui/ux"],
    # --- Web / Frontend
    "JavaScript": ["javascript", "js", "javascript (es6+)", "es6"],
    "TypeScript": ["typescript"],
    "React": ["react", "reactjs", "react.js", "react js"],
    "Vue.js": ["vue", "vuejs", "vue.js"],
    "Next.js": ["nextjs", "next.js"],
    "Node.js": ["node.js", "nodejs", "node js"],
    "Express.js": ["express", "expressjs", "express.js"],
    "NestJS": ["nestjs"],
    "HTML": ["html", "html5"],
    "CSS": ["css", "css3"],
    "jQuery": ["jquery"],
    "REST API": ["restful api", "restful apis", "rest api", "rest apis", "restful", "rest",
                 "api restful", "web api", "api", "apis", "api integration"],
    "WebSocket": ["websocket", "web socket"],
    "GraphQL": ["graphql"],
    # --- Backend / languages
    "Java": ["java", "java core"],
    "Spring Boot": ["spring boot", "springboot", "java spring boot"],
    "Spring": ["spring", "spring framework"],
    ".NET": [".net", ".net core", ".netcore", ".net framework", ".net mvc", ".net core mvc",
             ".net c#", ".net/c#", "asp.net", "asp.net core", "net core"],
    "C#": ["c#", "c-sharp"],
    "C++": ["c++"],
    "Go": ["go", "golang"],
    "PHP": ["php"],
    "Python": ["python"],
    "Kotlin": ["kotlin"],
    "Swift": ["swift"],
    "Dart": ["dart"],
    "Microservices": ["microservices", "microservice", "microservices architecture",
                      "microservice architecture"],
    "OOP": ["oop", "lập trình hướng đối tượng"],
    "Data Structures & Algorithms": ["cấu trúc dữ liệu", "giải thuật", "giải thuật và cấu trúc dữ liệu"],
    "Design Pattern": ["design pattern"],
    # --- Mobile
    "Android": ["android", "android studio", "android sdk"],
    "iOS": ["ios", "lập trình ios"],
    "React Native": ["react native"],
    "Flutter": ["flutter"],
    # --- Database
    "SQL": ["sql"],
    "SQL Server": ["sql server", "ms sql", "ms sql server", "sqlserver", "microsoft sql server"],
    "PostgreSQL": ["postgresql", "postgres", "postgre sql"],
    "MySQL": ["mysql"],
    "MongoDB": ["mongodb"],
    "NoSQL": ["nosql"],
    "PL/SQL": ["pl/sql", "plsql"],
    # --- DevOps / Cloud
    "Docker": ["docker"],
    "Kubernetes": ["kubernetes", "k8s", "kubernetes (k8s)"],
    "CI/CD": ["ci/cd", "cicd", "ci/cd pipelines", "ci/cd tools", "ci/cd pipeline"],
    "GitLab CI": ["gitlab ci", "gitlab ci/cd"],
    "GitHub Actions": ["github actions"],
    "Git": ["git"],
    "GitHub": ["github"],
    "GitLab": ["gitlab"],
    "ELK Stack": ["elk stack", "elk"],
    "AWS": ["aws", "amazon web services"],
    "Microsoft Azure": ["azure", "microsoft azure", "azure cloud"],
    "Google Cloud Platform": ["gcp", "google cloud", "google cloud platform"],
    "Bash": ["bash", "shell", "shell script"],
    "Windows Server": ["windows server"],
    "Linux": ["linux"],
    # --- Network / Infra
    "Networking": ["network", "networking", "mạng máy tính", "kiến thức về mạng máy tính",
                   "mạng lan/wan", "kiến thức nền tảng về network", "kiến thức nền tảng về mạng"],
    "Security": ["security", "bảo mật", "an ninh mạng", "an toàn thông tin"],
    # --- Data / AI
    "Machine Learning": ["machine learning", "ai/machine learning", "ml", "học máy", "ai và học máy"],
    "Deep Learning": ["deep learning"],
    "scikit-learn": ["scikit-learn", "sklearn"],
    "Power BI": ["power bi", "powerbi"],
    "Google Sheets": ["google sheets", "google sheet"],
    "Microsoft Office": ["microsoft office", "ms office", "microsoft 365", "word", "powerpoint",
                         "tin học văn phòng", "tin học văn phòng cơ bản", "thành thạo tin học văn phòng",
                         "tin học văn phòng (word, excel, powerpoint)", "tin học văn phòng (word, excel)",
                         "phần mềm văn phòng", "kỹ năng tin học văn phòng", "ms word"],
    "Data Analysis": ["phân tích dữ liệu", "data analysis", "data analytics"],
    # --- QA
    "Software Testing": ["kiểm thử phần mềm", "software testing", "testing", "manual testing"],
    "Test Automation": ["automation testing", "test automation", "automation", "automation test"],
    "API Testing": ["api testing", "kiểm thử api"],
    "Test Case": ["test case", "testcase", "viết testcase", "viết test case"],
    "Performance Testing": ["kiểm thử hiệu năng", "performance test", "performance testing"],
    # --- Process / Business
    "Agile": ["agile", "agile methodologies", "agile methodology"],
    "Scrum": ["scrum"],
    "Project Management": ["quản lý dự án", "project management", "quản trị dự án", "lập kế hoạch dự án",
                           "theo dõi tiến độ dự án"],
    "Business Analysis": ["phân tích nghiệp vụ", "phân tích yêu cầu", "business analysis"],
    "Product Management": ["quản lý sản phẩm", "product management"],
    "CRM": ["crm", "sử dụng crm"],
    "3D Modeling": ["3d modeling", "3d modelling", "3d"],
    "AI": ["ai", "ai tools", "trí tuệ nhân tạo", "ai trí tuệ nhân tạo"],
}
# 1 -> nhiều
MULTI_ALIAS = {
    "html/css": ["HTML", "CSS"],
    "c/c++": ["C", "C++"],
    "javascript/typescript": ["JavaScript", "TypeScript"],
    "agile/scrum": ["Agile", "Scrum"],
    "html, css": ["HTML", "CSS"],
}


##### Bước 3: Xây dựng class chuẩn hóa các kỹ năng

In [26]:
MIN_FREQ = 5  # skill chưa có alias phải xuất hiện >= 5 job mới được giữ
class SkillNormalizer:
    def __init__(self, raw_lists, min_freq=MIN_FREQ):
        self.min_freq = min_freq
        self.alias = {}
        for canon, variants in ALIAS_GROUPS.items():
            self.alias[key(canon)] = canon
            for v in variants:
                self.alias[key(v)] = canon
        # tần suất theo key (mỗi job đếm 1 lần)
        cnt, display = Counter(), {}
        for lst in raw_lists:
            seen = set()
            for s in lst:
                k = key(s)
                if k in seen:
                    continue
                seen.add(k)
                cnt[k] += 1
                display.setdefault(k, Counter())[s.strip()] += 1
        self.raw_freq = cnt
        # skill phổ biến chưa có alias -> tự thêm làm skill chuẩn
        self.vocab = set(ALIAS_GROUPS)
        for k, v in cnt.items():
            if v >= min_freq and k not in self.alias and k not in MULTI_ALIAS and not self._is_junk(k):
                canon = display[k].most_common(1)[0][0]
                self.alias[k] = canon
                self.vocab.add(canon)
        # regex dò từ vựng (chỉ canonical đủ dài / an toàn)
        terms = sorted({c for c in self.vocab if len(c) >= 4 or key(c) in SAFE_SHORT}, key=lambda c: (-len(c), c))
        self._scan = [(c, re.compile(r"(?<![\w+#.])" + re.escape(c.casefold()) + r"(?![\w+#])")) for c in terms]

    @staticmethod
    def _is_junk(k):
        return bool(SOFT_PREFIX.match(k)) or k in GENERIC or len(k) < 2 or len(k.split()) >= 6

    def _map_piece(self, k):
        if k in MULTI_ALIAS:
            return list(MULTI_ALIAS[k])
        if k in self.alias:
            return [self.alias[k]]
        return None

    def normalize_one(self, s):
        k = key(s)
        hit = self._map_piece(k)
        if hit is not None:
            return hit
        if self._is_junk(k) and not any(ch in k for ch in ",;:"):
            # câu mô tả thuần túy; vẫn cho dò từ vựng nếu có chứa skill rõ ràng
            pass
        out = []
        for piece in re.split(r"[,;:()]| và | and | & ", k):
            piece = piece.strip(" -.")
            if not piece:
                continue
            hit = self._map_piece(piece)
            if hit is not None:
                out += hit
                continue
            # dò từ vựng bên trong mảnh
            for canon, rx in self._scan:
                if rx.search(piece):
                    out.append(canon)
        return out

    def transform(self, lst):
        res, seen = [], set()
        for s in lst:
            for c in self.normalize_one(s):
                if c not in seen:
                    seen.add(c)
                    res.append(c)
        return res

##### Bước 4: Chuẩn hóa từng tin tuyển dụng
Khởi tạo normalizer từ class ở bước 3 trên và áp dụng trên toàn bộ dataset


In [27]:
all_raw_skills= df["skills_list"].to_list()

normalizer = SkillNormalizer(all_raw_skills, MIN_FREQ)

print(f"Tổng số kỹ năng chuẩn đã học được: {len(normalizer.vocab)}")
print(f"Tổng số biến thể alias: {len(normalizer.alias)}")

Tổng số kỹ năng chuẩn đã học được: 199
Tổng số biến thể alias: 350


In [28]:
# Áp dụng hàm transform lên từng dòng
df["skills_clean"] = df["skills_list"].apply(normalizer.transform)

# Xem kết quả
print(df[["skills_list", "skills_clean"]].head(10))

                                         skills_list  \
0  [MCSA, MCSE, Vmware, LPIC, Netapp, Cisco, HPE,...   
1  [ReactJS, NextJS, JavaScript, Redux, Recoil, N...   
2  [Java, NodeJS, AWS cloud platform, microservic...   
3                                 [Tiếng Anh cơ bản]   
4  [Adobe Illustrator, Adobe InDesign, Adobe Phot...   
5     [Node.js, Express, NestJS, MySQL, CassandraDB]   
6  [C#, .NET, HTML5, JavaScript, WCF, REST-full, ...   
7  [HTML5, CSS3, JavaScript, Ajax, JSON, Fetch, j...   
8  [.NET C#, HTML5, CSS3, JavaScript, JQuery, Kno...   
9  [Excel nâng cao, Google Sheets, SQL, Power BI,...   

                                        skills_clean  
0                           [VMware, Cisco, Storage]  
1  [React, Next.js, JavaScript, Redux, Node.js, W...  
2  [Java, Node.js, Cloud, AWS, Microservices, Doc...  
3                                                 []  
4                 [Illustrator, InDesign, Photoshop]  
5               [Node.js, Express.js, NestJS, MySQL] 

Ngoài ra dataset này còn có requirements_text hỗ trợ đáng kể với việc đề xuất kĩ năng cần học với job, 
cần xây dựng thêm hàm đọc từ text dò technical_skils

In [29]:
print(df["requirements_text"])

0       Yêu cầu chung: Tốt nghiệp ĐH chuyên ngành CNTT...
1       WHAT WE ARE LOOKING FOR: Education: Bachelor’s...
2       Proven experience (8+ years) in software devel...
3       - Ưu tiên có kinh nghiệm về game online, làm l...
4       Biết sử dụng máy tính. Sẽ được hướng dẫn và đà...
                              ...                        
1901    1. Trình độ: Tốt nghiệp đại học chuyên ngành C...
1902    Có background về Data Analyst, Business Analys...
1903    Có kinh nghiệm tối thiểu 2 năm ở vị trí tương ...
1904    1. Ưu tiên Nam, tốt nghiệp cao đẳng trở lên (c...
1905    Bằng cấp: Trung cấp - cao đẳng - đại học (chấp...
Name: requirements_text, Length: 1906, dtype: str


##### Bước 5.1: Xây dừng các hàm kiểm tra ascii, scan text, và lấy kĩ năng từ text

In [30]:
def _ascii(s):
    return all(ord(ch) < 128 for ch in s)


def build_text_scanner(normalizer):
    """Từ vựng ASCII (tên công nghệ) dùng để dò trong requirements_text."""
    terms = sorted({c for c in normalizer.vocab if _ascii(c) and (len(c) >= 4 or key(c) in SAFE_SHORT)},
                   key=lambda c: (-len(c), c))
    return [(c, re.compile(r"(?<![\w+#.])" + re.escape(c.casefold()) + r"(?![\w+#])")) for c in terms]


def enrich_from_text(text, scanner):
    t = str(text).casefold()
    return [c for c, rx in scanner if rx.search(t)]

##### Bước 5.2: Viết hàm áp dụng 3 hàm trên lên cột requirements_text

In [31]:
# Khởi tạo bộ quét văn bản từ normalizer hiện tại
text_scanner = build_text_scanner(normalizer)
def extract_all_skills(row):
    # 1. Lấy kỹ năng đã chuẩn hóa từ cột skills_list
    skills_from_list = row["skills_clean"] 
    
    # 2. Trích xuất kỹ năng từ cột requirement (nếu có)
    desc_text = row.get("requirements_text", "") # Lấy nội dung cột requirement
    
    # Nếu requirement bị trống (NaN) thì trả về mảng rỗng
    if pd.isna(desc_text):
        desc_text = ""
        
    # Dùng text_scanner để dò tìm kỹ năng ẩn trong đoạn văn
    skills_from_text = enrich_from_text(desc_text, text_scanner)
    
    # 3. Gộp 2 danh sách và loại bỏ trùng lặp (giữ nguyên thứ tự)
    combined_skills = []
    seen = set()
    
    # Ưu tiên kỹ năng từ list trước
    for s in skills_from_list + skills_from_text:
        if s not in seen:
            seen.add(s)
            combined_skills.append(s)
            
    return combined_skills

# Áp dụng hàm lên toàn bộ dataset (chạy theo từng dòng nên dùng axis=1)
df["skills_final"] = df.apply(extract_all_skills, axis=1)

##### Bước 5.3: Xem bao nhiêu skill được bổ sung từ requirements_text

In [32]:
# Xem thử 5 dòng
print(df[["skills_clean", "skills_final"]].head(5))

# Đếm xem cột requirements_text đóng góp thêm bao nhiêu kỹ năng mới
for i in range(5):
    added = set(df["skills_final"].iloc[i]) - set(df["skills_clean"].iloc[i])
    if added:
        print(f"Dòng {i} - Tìm thấy thêm từ Description: {added}")

                                        skills_clean  \
0                           [VMware, Cisco, Storage]   
1  [React, Next.js, JavaScript, Redux, Node.js, W...   
2  [Java, Node.js, Cloud, AWS, Microservices, Doc...   
3                                                 []   
4                 [Illustrator, InDesign, Photoshop]   

                                        skills_final  
0                           [VMware, Cisco, Storage]  
1  [React, Next.js, JavaScript, Redux, Node.js, W...  
2  [Java, Node.js, Cloud, AWS, Microservices, Doc...  
3                                                 []  
4                 [Illustrator, InDesign, Photoshop]  
Dòng 1 - Tìm thấy thêm từ Description: {'Jest', 'CI/CD', 'Scrum', 'Agile', 'Kafka', 'Elasticsearch'}


### 3.3 Gom nhóm job_title
Từ 1328 giá trị khác nhau như ở trên sẽ gom thành ~12 role

Chỉ dùng job_title (KHÔNG dùng skills) để nhãn role độc lập với thứ cần dự đoán.

Thứ tự regex = thứ tự ưu tiên (khớp đầu tiên là điểm cao)

In [33]:
import re

R = lambda p: re.compile(p, re.I)

OTHER = "Other"
GAME_ARTIST = R(r"artist|designer|họa sĩ|đồ họa|animator|\b3d\b|\b2d\b|vfx|concept|thiết kế")

ROLE_RULES = [
    ("DevOps / Infra / Security", R(r"trung tâm dữ liệu|data ?cent(er|re)")),
    ("QA / Tester", R(r"\bqa\b|\bqc\b|test|kiểm thử|thử nghiệm|kiểm tra phần mềm|kiểm tra tự động|đảm bảo chất lượng|"
                      r"quality assurance|quản lý chất lượng phần mềm|kiểm tra hiệu suất")), # Thêm "thử nghiệm", "kiểm tra hiệu suất"
    ("BA / PM / Product", R(r"business analyst|\bba\b|phân tích nghiệp vụ|phân tích kinh doanh|phân tích hệ thống|system analyst|"
                           r"\bpm\b|project manager|product|quản lý dự án|quản trị dự án|scrum|\bpo\b|chủ sở hữu sản phẩm|"
                           r"delivery|presale|pre-sale|quản lý sản phẩm|tư vấn viên|consultant|chuyên viên tư vấn|"
                           r"giải pháp|solution|triển khai|implementation|quản lý kỹ thuật|trưởng dự án|"
                           r"cải tiến quy trình|giám đốc dự án|\bpmo\b|producer|tăng trưởng sản phẩm|chuyển đổi số|"
                           r"tư vấn|đảm bảo quy trình|trợ lý dự án|sản phẩm|dự án|lãnh đạo kỹ thuật|tech lead|điều hành|erp")), # Thêm "sản phẩm", "dự án", "tech lead", "điều hành", "erp"
    ("Data / AI", R(r"\bdata\b|dữ liệu|\bai\b|machine learning|\bml\b|\bllm\b|\bnlp\b|power bi|\bbi\b|trí tuệ nhân tạo|học máy|"
                    r"phân tích|analyst|analytics|nhãn dữ liệu")),
    ("DevOps / Infra / Security", R(r"devops|devsecops|\bsre\b|cloud|đám mây|hệ thống|system|infra|hạ tầng|network|mạng|"
                                    r"vận hành|security|bảo mật|an ninh|an toàn thông tin|helpdesk|help desk|it support|"
                                    r"\bdba\b|quản trị|sysadmin|\bnoc\b|kỹ thuật viên|công nghệ thông tin|\bit\b|phần cứng|"
                                    r"kỹ thuật máy tính|camera")),
    ("Mobile Dev", R(r"mobile|android|\bios\b|flutter|react native|swift|kotlin|di động")),
    ("Frontend Dev", R(r"front.?end|\breact|angular|\bvue|web ?developer|\bhtml|giao diện|ui developer|javascript|\bjs\b")),
    ("Backend / Fullstack Dev", R(r"back.?end|full.?stack|java\b|\.net|dotnet|\bnode|python|php|golang|\bgo\b|developer|lập trình|"
                                  r"phát triển phần mềm|software|engineer|kỹ sư phần mềm|\bc#|c\+\+|c/c\+\+|nhúng|spring|laravel|programmer|" # Thêm "c/c++", "nhúng"
                                  r"embedded|firmware|nhà phát triển|\bapi\b|\bsap\b|abap|odoo|\bdev\b|kỹ sư|tech lead|"
                                  r"team lead|kiến trúc sư giải pháp|architect|\bde\b")),
    ("Designer / Creative", R(r"thiết kế|design|đồ họa|graphic|\bui\b|\bux\b|artist|animator|motion|video|editor|dựng|"
                              r"\b3d\b|\b2d\b|creative|illustrat|họa sĩ|concept|vfx|revit|\bcad\b|autocad|nội thất|diễn họa|"
                              r"nhiếp ảnh|photo|kiến trúc|sketchup|lumion|render|kỹ thuật viên đồ họa|hình ảnh|"
                              r"bim|diễn hoạt|in ấn|bình ghép file|mep")), # Thêm "bim", "diễn hoạt", "in ấn", v.v.
    ("Sales / Marketing / Support", R(r"kinh doanh|sales|\bsale\b|marketing|bán hàng|account|support|hỗ trợ|chăm sóc|tuyển dụng|"
                                      r"nhân sự|content|\bseo\b|admin|văn phòng|kế toán|hành chính|truyền thông|thương mại|"
                                      r"tiếp thị|khách hàng|quảng cáo|thu mua|\bbd\b|telesale|\bcsr\b|\bhr\b|"
                                      r"business development|phát triển thị trường|quản lý tài khoản|thực tập")), # Thêm "business development", v.v.
]
GAME_DEV = R(r"game|unity|unreal")


def title_to_role(title: str) -> str:
    t = str(title).strip()
    if GAME_DEV.search(t) and not GAME_ARTIST.search(t):
        return "Game Dev"
    if re.search(r"thiết kế|designer|đồ họa|graphic|ui/ux|\bux\b", t, re.I) and not re.search(
            r"hệ thống|system|phần mềm|software|database|cơ sở dữ liệu|network|mạng|giải pháp", t, re.I):
        return "Designer / Creative"
    for role, rx in ROLE_RULES:
        if rx.search(t):
            return role
    return OTHER

In [34]:
# Áp dụng hàm phân loại vào từng dòng
df['role_group'] = df['job_title'].apply(title_to_role)

# Kiểm tra kết quả xem đã gom được chưa
print("=== PHÂN BỐ CÁC NHÓM NGÀNH NGHỀ ===")
print(df['role_group'].value_counts())

# Xem thử vài ví dụ để kiểm tra độ chính xác
print("\n=== VÍ DỤ KẾT QUẢ ===")
print(df[['job_title', 'role_group']].sample(10, random_state=42))

=== PHÂN BỐ CÁC NHÓM NGÀNH NGHỀ ===
role_group
Designer / Creative            435
BA / PM / Product              357
Backend / Fullstack Dev        259
DevOps / Infra / Security      245
Sales / Marketing / Support    215
Data / AI                       95
QA / Tester                     90
Other                           76
Frontend Dev                    63
Mobile Dev                      50
Game Dev                        21
Name: count, dtype: int64

=== VÍ DỤ KẾT QUẢ ===
                                           job_title  \
1226                      Thực Tập Sinh IT Help Desk   
111               Chuyên Viên Kỹ Thuật Vận Hành VoIP   
1839                            Giám đốc PMO cao cấp   
1675        Bán hàng Đại diện B2B (Fresher - Junior)   
415                               Nhân Viên Thiết Kế   
1366                             Chủ sở hữu sản phẩm   
70                     Nhân Viên Triển Khai Phần Mềm   
530                                  Lập Trình Nhúng   
617   Người mô 

In [35]:
# Lọc ra các job bị rơi vào Other
other_jobs = df[df['role_group'] == 'Other']['job_title'].unique()

print(f"Có {len(other_jobs)} title đang bị xếp vào nhóm Other:")
for job in other_jobs[:30]:  # In thử 30 cái đầu
    print(f"- {job}")

Có 67 title đang bị xếp vào nhóm Other:
- Chuyên Viên Kiểm Soát Nội Bộ
- Nhân Viên Bảo Trì Điện (PLC - MÁY CNC)
- Trợ Lý Giám Đốc
- Phó Trưởng Phòng Sản Phầm Dịch Vụ Phần Mềm
- Nhân Viên Phụ Bếp
- Chuyên Viên ESG
- Nhân Viên Tái Ký Hợp Đồng
- SOC Tier 2
- Quản Lý Kĩ Thuật
- Nhà lãnh đạo xã hội đầy đủ
- Giám đốc công nghệ
- Nó lãnh đạo hoạt động
- Chuyên Viên Vẽ Minh Họa (Character/Pattern)
- Nhân Viên Tin Học
- Chuyên gia quản lý SOC
- Chief Technology Officer ( Ứng Dụng SAAS)
- Agile Coach
- Học tập củng cố
- Nhân Viên Media
- Trưởng Nhóm Bảo Trì Điện PLC
- Trình quản lý dịch vụ CNTT
- LiveOps gd
- Trợ Lý Chủ Tịch
- Trưởng Nhóm Bảo Trì Điện PLC (Nhà Máy)
- Nhân Viên Idea Creator
- CTV Kỹ Thuật Viễn Thông
- Chuyên Viên Cao Cấp Quản Lý Sự Cố Và Vấn Đề
- Nhân Viên Kỹ Thuật
- Kỹ thuật dẫn đầu
- Nhân Viên CNC


### 3.4 Chuẩn hóa kinh nghiệm thành 4 cấp bậc

Cột `experience_required` là chuỗi tự do ("Không yêu cầu", "6 tháng", "2 năm", ...). Mục này đổi chuỗi sang số tháng rồi chia thành 4 cấp bậc để dùng làm ngữ cảnh cho luật kết hợp ở hai bài toán sau:

| Cấp bậc | Quy tắc theo số tháng kinh nghiệm yêu cầu |
|---|---|
| Intern | 0 (Không yêu cầu) |
| Fresher | trên 0 đến 12 tháng |
| Junior | trên 12 đến 36 tháng (2–3 năm) |
| Senior | trên 36 tháng (từ 4 năm) |

**Lưu ý:** cấp bậc ở đây chỉ là *proxy* suy ra từ yêu cầu kinh nghiệm của tin tuyển dụng. `Intern` nghĩa là tin không đòi hỏi kinh nghiệm (gồm cả thực tập sinh lẫn vị trí nhận người mới), không phải nhãn lấy từ chức danh. Ranh giới giữa các cấp là lựa chọn của người phân tích và có thể chỉnh ở hàm `months_to_level`.

In [36]:
# Các giá trị thô của cột kinh nghiệm
print("Số dòng thiếu:", df["experience_required"].isna().sum())
df["experience_required"].value_counts(dropna=False)

Số dòng thiếu: 0


experience_required
2 năm            479
1 năm            437
3 năm            357
Không yêu cầu    341
5 năm            123
6 tháng           79
4 năm             64
6 năm              7
7 năm              7
3 tháng            4
8 năm              3
18 tháng           2
2 tháng            1
30 tháng           1
10 năm             1
Name: count, dtype: int64

In [37]:
import numpy as np

EXP_LEVELS = ["Intern", "Fresher", "Junior", "Senior"]

def parse_experience_months(value):
    # '2 năm' -> 24, '6 tháng' -> 6, 'Không yêu cầu' -> 0; không đọc được -> NaN
    s = str(value).strip().casefold()
    if s in ("", "nan", "none"):
        return np.nan
    if "không yêu cầu" in s or "không cần" in s:
        return 0.0
    m = re.search(r"(\d+(?:[.,]\d+)?)", s)
    if not m:
        return np.nan
    n = float(m.group(1).replace(",", "."))
    return n if "tháng" in s else n * 12

def months_to_level(months):
    if pd.isna(months): return "Unknown"
    if months == 0:     return "Intern"
    if months <= 12:    return "Fresher"
    if months <= 36:    return "Junior"
    return "Senior"

df["exp_months"] = df["experience_required"].apply(parse_experience_months)
df["exp_level"] = df["exp_months"].apply(months_to_level)

# Kiểm tra ánh xạ từng giá trị thô -> cấp bậc
mapping_check = (df.groupby(["experience_required", "exp_months", "exp_level"]).size()
                 .rename("Số tin").reset_index().sort_values("exp_months"))
mapping_check

,experience_required,exp_months,exp_level,Số tin
14,Không yêu cầu,0.0,Intern,341
4,2 tháng,2.0,Fresher,1
6,3 tháng,3.0,Fresher,4
11,6 tháng,6.0,Fresher,79
0,1 năm,12.0,Fresher,437
2,18 tháng,18.0,Junior,2
3,2 năm,24.0,Junior,479
7,30 tháng,30.0,Junior,1
5,3 năm,36.0,Junior,357
8,4 năm,48.0,Senior,64


In [38]:
df_known = df[df["role_group"] != "Other"].copy()
df_known["n_skills"] = df_known["skills_final"].map(len)

print("=== Phân bố cấp bậc (đã bỏ Other) ===")
print(df_known["exp_level"].value_counts().reindex(EXP_LEVELS + ["Unknown"]).dropna().astype(int))

print("\n=== Số role x cấp bậc ===")
print(pd.crosstab(df_known["role_group"], df_known["exp_level"]).reindex(columns=EXP_LEVELS, fill_value=0))

print("\n=== Số skill trung bình mỗi tin theo cấp bậc ===")
print(df_known.groupby("exp_level")["n_skills"].mean().reindex(EXP_LEVELS).round(2))

=== Phân bố cấp bậc (đã bỏ Other) ===
exp_level
Intern     328
Fresher    505
Junior     806
Senior     191
Name: count, dtype: int64

=== Số role x cấp bậc ===
exp_level                    Intern  Fresher  Junior  Senior
role_group                                                  
BA / PM / Product                51       75     166      65
Backend / Fullstack Dev          34       46     140      39
Data / AI                        17       16      49      13
Designer / Creative              76      186     156      17
DevOps / Infra / Security        42       66     122      15
Frontend Dev                      3        8      41      11
Game Dev                          4       11       6       0
Mobile Dev                        1       15      25       9
QA / Tester                      12       19      48      11
Sales / Marketing / Support      88       63      53      11

=== Số skill trung bình mỗi tin theo cấp bậc ===
exp_level
Intern     1.96
Fresher    3.54
Junior     5.55

**Nhận xét**

- Cấp bậc càng cao thì tin càng liệt kê nhiều kỹ năng kỹ thuật; sự khác biệt rõ nhất nằm giữa Fresher và Junior.
- Dữ liệu phân bố không đều giữa các ô role × cấp bậc: nhiều role chỉ có rất ít tin Senior hoặc Intern. Vì vậy ở bài toán 1 và 2, luật theo ô (role, cấp bậc) chỉ được dùng khi ô đủ lớn, ô nhỏ sẽ quay về luật theo role (backoff).

In [39]:
# Trong cùng một role, kỹ năng nào nổi bật hơn ở Senior so với Intern/Fresher? (chỉ xét role đủ dữ liệu)
from collections import Counter

base = df_known[df_known["skills_final"].map(bool)]
rows = []
for role, g in base.groupby("role_group"):
    low = g[g["exp_level"].isin(["Intern", "Fresher"])]
    high = g[g["exp_level"] == "Senior"]
    if len(low) < 20 or len(high) < 20:
        continue
    c_low = Counter(s for l in low["skills_final"] for s in l)
    c_high = Counter(s for l in high["skills_final"] for s in l)
    diff = {s: c_high[s] / len(high) - c_low[s] / len(low)
            for s in set(c_low) | set(c_high) if c_low[s] + c_high[s] >= 10}
    ranked = sorted(diff.items(), key=lambda kv: kv[1])
    rows.append({"role_group": role, "Intern+Fresher": len(low), "Senior": len(high),
                 "Skill nổi bật ở Senior": ", ".join(f"{s} ({v:+.0%})" for s, v in ranked[::-1][:3]),
                 "Skill nổi bật ở Intern/Fresher": ", ".join(f"{s} ({v:+.0%})" for s, v in ranked[:3])})
pd.DataFrame(rows)

,role_group,Intern+Fresher,Senior,Skill nổi bật ở Senior,Skill nổi bật ở Intern/Fresher
0,BA / PM / Product,71,60,"Cloud (+19%), Agile (+16%), REST API (+14%)","Microsoft Office (-19%), Excel (-10%), ERP (-4%)"
1,Backend / Fullstack Dev,70,39,"AWS (+27%), SQL (+23%), Microservices (+22%)","C# (-19%), Python (-16%), JavaScript (-12%)"


### 3.5 Transaction & One-Hot Encoding

Role mới là cặp `{role, cấp bậc kinh nghiệm}` được gộp thành **một item** `ROLE_<role> | <cấp bậc>`, ví dụ `ROLE_Backend / Fullstack Dev | Senior`. Mỗi dòng dữ liệu là một transaction `[ROLE_x | EXP, skill, skill, ...]`; hai bài toán sau dùng nguyên cách làm cũ với role mới này. Tin thuộc nhóm `Other` giữ nguyên nhãn `Other` và bị loại như trước.

In [40]:
# Role mới = {role, cấp bậc kinh nghiệm} gộp thành một nhãn; Other giữ nguyên để vẫn bị loại
df["role_exp"] = df["role_group"].where(df["role_group"] == "Other", df["role_group"] + " | " + df["exp_level"])

# Tạo danh sách giao dịch: [ROLE_Frontend Dev | Junior, React, JavaScript, ...]
transactions = []
for _, row in df.iterrows():
    # Bỏ qua nếu role là Other (rác)
    if row['role_exp'] == 'Other':
        continue 
    
    # Tạo list gồm Role mới + toàn bộ skills
    item = [f"ROLE_{row['role_exp']}"] + row['skills_final']
    transactions.append(item)

# Mã hóa One-hot
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)

# Lưu thành DataFrame
transaction_df = pd.DataFrame(te_ary, columns=te.columns_)
print("Kích thước ma trận giao dịch:", transaction_df.shape)
print("Số role mới (role x cấp bậc):", sum(c.startswith("ROLE_") for c in transaction_df))

Kích thước ma trận giao dịch: (1830, 239)
Số role mới (role x cấp bậc): 39


## 4. Lưu tập dữ liệu đã xử lý và ma trận giao dịch

In [41]:
df.to_csv(
    "../data/processed/processed_jobs.csv",
    index=False,
    encoding = "utf-8-sig"
)

In [42]:
df[["skills_list", "skills_clean", "skills_final"]].to_csv(
    "../data/processed/processed_skills.csv",
    index=False,
    encoding="utf-8-sig"
)

In [43]:
transaction_df.to_csv(
    "../data/processed/transaction_matrix.csv",
    index=False
)